# Backtest Engine Template

A creative, modular notebook starter for research-grade strategy testing.

## What this template includes
- Config-driven setup
- Synthetic and CSV market data loaders
- Signal + portfolio + execution separation
- Metrics, tearsheet table, and equity chart
- Parameter sweep scaffold
- Walk-forward analysis scaffold

> Tip: replace the toy strategy with your own alpha logic while keeping the interfaces intact.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")
pd.options.display.float_format = '{:,.4f}'.format

In [ ]:
@dataclass
class BacktestConfig:
    symbol: str = "SYNTH"
    frequency: str = "D"
    initial_capital: float = 100_000.0
    fee_bps: float = 1.0
    slippage_bps: float = 2.0
    annualization: int = 252
    seed: int = 42

    # Toy strategy defaults (moving-average crossover)
    fast_window: int = 20
    slow_window: int = 100

cfg = BacktestConfig()
cfg

## 1) Data layer

Choose one: generate synthetic prices for prototyping or load real OHLCV data from CSV.

In [ ]:
def generate_synthetic_data(n: int = 1500, start: str = "2018-01-01", seed: int = 42) -> pd.DataFrame:
    """Geometric random walk with weak regime shifts and noisy volume."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start=start, periods=n, freq="B")

    # three drift regimes
    drift = np.concatenate([
        np.full(n // 3, 0.0002),
        np.full(n // 3, -0.0001),
        np.full(n - 2 * (n // 3), 0.00035),
    ])
    vol = 0.012
    rets = drift + rng.normal(0, vol, size=n)

    close = 100 * np.exp(np.cumsum(rets))
    open_ = close * (1 + rng.normal(0, 0.001, size=n))
    high = np.maximum(open_, close) * (1 + np.abs(rng.normal(0, 0.0018, size=n)))
    low = np.minimum(open_, close) * (1 - np.abs(rng.normal(0, 0.0018, size=n)))
    volume = rng.integers(200_000, 1_000_000, size=n)

    df = pd.DataFrame({
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
    }, index=dates)
    df.index.name = "date"
    return df

def load_csv_data(path: str, date_col: str = "date") -> pd.DataFrame:
    df = pd.read_csv(path)
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    required = {"open", "high", "low", "close"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if "volume" not in df.columns:
        df["volume"] = np.nan
    return df

market = generate_synthetic_data(seed=cfg.seed)
market.head()

## 2) Alpha layer (signals)

Signal convention used in this template:
- +1: fully long
-  0: flat
- -1: fully short

In [ ]:
def moving_average_crossover_signal(close: pd.Series, fast: int, slow: int) -> pd.Series:
    fast_ma = close.rolling(fast).mean()
    slow_ma = close.rolling(slow).mean()
    signal = np.where(fast_ma > slow_ma, 1, -1)
    signal = pd.Series(signal, index=close.index, name="target_position")
    signal = signal.where(fast_ma.notna() & slow_ma.notna(), 0)
    return signal

target_position = moving_average_crossover_signal(
    market["close"],
    fast=cfg.fast_window,
    slow=cfg.slow_window,
)
target_position.tail()

## 3) Execution + portfolio accounting

The engine below keeps things intentionally simple while preserving realistic structure:
- next-bar execution via lagged position
- transaction cost = turnover × (fees + slippage)
- equity curve compounded from daily net returns

In [ ]:
def run_backtest(
    market: pd.DataFrame,
    target_position: pd.Series,
    cfg: BacktestConfig,
) -> pd.DataFrame:
    df = market.copy()
    df["target_position"] = target_position.reindex(df.index).fillna(0).clip(-1, 1)

    # Execute next bar to avoid look-ahead bias
    df["position"] = df["target_position"].shift(1).fillna(0)
    df["asset_return"] = df["close"].pct_change().fillna(0)
    df["gross_return"] = df["position"] * df["asset_return"]

    turnover = df["position"].diff().abs().fillna(df["position"].abs())
    total_cost_rate = (cfg.fee_bps + cfg.slippage_bps) / 10_000
    df["cost_return"] = turnover * total_cost_rate
    df["net_return"] = df["gross_return"] - df["cost_return"]

    df["equity"] = cfg.initial_capital * (1 + df["net_return"]).cumprod()
    df["peak_equity"] = df["equity"].cummax()
    df["drawdown"] = df["equity"] / df["peak_equity"] - 1
    return df

results = run_backtest(market, target_position, cfg)
results[["close", "position", "equity", "drawdown"]].tail()

## 4) Evaluation metrics

In [ ]:
def summarize_performance(bt: pd.DataFrame, annualization: int = 252) -> pd.Series:
    r = bt["net_return"].dropna()
    eq = bt["equity"]

    total_return = eq.iloc[-1] / eq.iloc[0] - 1
    years = max(len(r) / annualization, 1 / annualization)
    cagr = (1 + total_return) ** (1 / years) - 1

    vol = r.std() * np.sqrt(annualization)
    sharpe = (r.mean() * annualization) / vol if vol > 0 else np.nan

    downside = r[r < 0].std() * np.sqrt(annualization)
    sortino = (r.mean() * annualization) / downside if downside > 0 else np.nan

    max_dd = bt["drawdown"].min()
    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan

    win_rate = (r > 0).mean()

    stats = pd.Series({
        "Total Return": total_return,
        "CAGR": cagr,
        "Annual Volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Calmar": calmar,
        "Max Drawdown": max_dd,
        "Win Rate": win_rate,
        "Avg Daily Return": r.mean(),
        "Avg Daily Turnover": bt["position"].diff().abs().mean(),
    })
    return stats

summary = summarize_performance(results, annualization=cfg.annualization)
summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(results.index, results["equity"], label="Strategy equity", lw=2)
axes[0].set_title(f"{cfg.symbol} Strategy Equity Curve")
axes[0].legend(loc="upper left")

axes[1].fill_between(results.index, results["drawdown"], 0, color="crimson", alpha=0.3)
axes[1].set_title("Drawdown")

plt.tight_layout()
plt.show()

## 5) Parameter sweep scaffold

Grid-search moving-average parameters and rank by Sharpe.

In [ ]:
def parameter_sweep(
    market: pd.DataFrame,
    cfg: BacktestConfig,
    fast_grid: List[int],
    slow_grid: List[int],
) -> pd.DataFrame:
    rows = []
    for fast in fast_grid:
        for slow in slow_grid:
            if fast >= slow:
                continue
            sig = moving_average_crossover_signal(market["close"], fast=fast, slow=slow)
            bt = run_backtest(market, sig, cfg)
            stats = summarize_performance(bt, annualization=cfg.annualization)
            rows.append({
                "fast": fast,
                "slow": slow,
                "Sharpe": stats["Sharpe"],
                "CAGR": stats["CAGR"],
                "Max Drawdown": stats["Max Drawdown"],
            })

    out = pd.DataFrame(rows).sort_values("Sharpe", ascending=False).reset_index(drop=True)
    return out

grid_results = parameter_sweep(market, cfg, fast_grid=[10, 20, 30, 40], slow_grid=[80, 100, 120, 150])
grid_results.head(10)

## 6) Walk-forward analysis scaffold

Use rolling train/test windows to reduce overfitting from static parameter selection.

In [ ]:
def walk_forward_template(
    market: pd.DataFrame,
    cfg: BacktestConfig,
    train_size: int = 500,
    test_size: int = 125,
) -> pd.DataFrame:
    folds = []
    start = 0

    while start + train_size + test_size <= len(market):
        train = market.iloc[start : start + train_size]
        test = market.iloc[start + train_size : start + train_size + test_size]

        # Pick best params from training window
        train_grid = parameter_sweep(
            train,
            cfg,
            fast_grid=[10, 20, 30],
            slow_grid=[80, 100, 120],
        )
        best = train_grid.iloc[0]

        # Evaluate on out-of-sample test window
        sig = moving_average_crossover_signal(test["close"], int(best.fast), int(best.slow))
        bt = run_backtest(test, sig, cfg)
        stats = summarize_performance(bt, annualization=cfg.annualization)

        folds.append({
            "start": test.index.min(),
            "end": test.index.max(),
            "fast": int(best.fast),
            "slow": int(best.slow),
            "Sharpe": stats["Sharpe"],
            "CAGR": stats["CAGR"],
            "Max Drawdown": stats["Max Drawdown"],
        })

        start += test_size

    return pd.DataFrame(folds)

wf_results = walk_forward_template(market, cfg)
wf_results

## 7) Next upgrades

- Multi-asset portfolio construction (risk parity, vol targeting, beta neutral)
- Event-driven engine with order objects and fill simulation
- Benchmark-relative attribution and factor decomposition
- Integration with hyperparameter optimization (Optuna/Ray Tune)
- Production handoff: package into modules and add unit tests